<a href="https://colab.research.google.com/github/rehunter3502026-max/diabetes-detection/blob/main/v1_prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================================
# CELL 1 : IMPORT LIBRARIES
# ==========================================================

# -------------------------
# Data Manipulation
# -------------------------
import pandas as pd
import numpy as np

# -------------------------
# Operating System Utilities
# -------------------------
import os
import re
import string
import warnings

# -------------------------
# Date & Time
# -------------------------
from datetime import datetime

# -------------------------
# Natural Language Processing
# -------------------------
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# -------------------------
# Machine Learning
# -------------------------
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# -------------------------
# Save ML Objects
# -------------------------
import joblib

# -------------------------
# Display Settings
# -------------------------
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

warnings.filterwarnings("ignore")

# -------------------------
# Download NLTK Resources
# -------------------------
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

print("=" * 60)
print("All Libraries Imported Successfully")
print("=" * 60)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


All Libraries Imported Successfully


[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
# ==========================================================
# CELL 2 : LOAD DATASET
# ==========================================================

from google.colab import files

print("="*60)
print("UPLOAD HOSPITAL DATASET")
print("="*60)

uploaded = files.upload()

DATASET_PATH = list(uploaded.keys())[0]

df = pd.read_csv(DATASET_PATH)

print("\nDataset Loaded Successfully!")

print("="*60)
print("Dataset Shape")
print("="*60)

print(df.shape)

print("="*60)
print("First Five Records")
print("="*60)

display(df.head())

UPLOAD HOSPITAL DATASET


Saving hospital_records_2021_2024_with_bills.csv to hospital_records_2021_2024_with_bills (1).csv

Dataset Loaded Successfully!
Dataset Shape
(1000, 10)
First Five Records


,Patient ID,Name,Date of Birth,Gender,Medical Condition,Treatments,Doctor's Notes,Admit Date,Discharge Date,Bill Amount
0,921d93d1-17b8-426f-abae-5b16c7e5cd93,Debra Griffith,1971-04-01,Female,Chronic Obstructive Pulmonary Disease,Medication,Patient requires home oxygen therapy.,2021-07-12,2021-07-13,14651.34
1,c3e70ecd-b794-4e9e-8136-ccf4ff363fe0,Robert Ross,1957-02-20,Female,Migraine,Pain Relief Medication,Follow-up to assess effectiveness of treatment.,2021-07-13,2021-08-09,718.94
2,ef85177f-f933-4f60-aefe-6979e35c709a,Rachel Perez,1987-11-27,Female,Urinary Tract Infection,Hydration,Monitor for any signs of worsening symptoms.,2021-07-14,2021-08-06,1765.14
3,8e36ba50-ebde-4b68-81b9-3cdca0b7eade,Michael Williams,2021-12-06,Male,Stroke,Physical Therapy,Referred to physical therapy for mobility improvement.,2021-07-15,2021-08-05,21413.36
4,422bd94a-41e1-4bdc-96a3-f7570a868df1,Christopher Rodriguez,1990-12-19,Male,Hypertension,Lifestyle Changes,Patient started on antihypertensive medication.,2021-07-16,2021-07-20,4879.91


In [ ]:
# ==========================================================
# CELL 3 : DATASET EXPLORATION
# ==========================================================

print("="*60)
print("COLUMN NAMES")
print("="*60)

print(df.columns.tolist())

print("\n")

print("="*60)
print("DATA TYPES")
print("="*60)

display(df.dtypes)

print("\n")

print("="*60)
print("MISSING VALUES")
print("="*60)

display(df.isnull().sum())

print("\n")

print("="*60)
print("DUPLICATE RECORDS")
print("="*60)

print(df.duplicated().sum())

print("\n")

print("="*60)
print("DATASET INFORMATION")
print("="*60)

df.info()

COLUMN NAMES
['Patient ID', 'Name', 'Date of Birth', 'Gender', 'Medical Condition', 'Treatments', "Doctor's Notes", 'Admit Date', 'Discharge Date', 'Bill Amount']


DATA TYPES


,0
Patient ID,object
Name,object
Date of Birth,object
Gender,object
Medical Condition,object
Treatments,object
Doctor's Notes,object
Admit Date,object
Discharge Date,object
Bill Amount,float64




MISSING VALUES


,0
Patient ID,0
Name,0
Date of Birth,0
Gender,0
Medical Condition,0
Treatments,0
Doctor's Notes,0
Admit Date,0
Discharge Date,0
Bill Amount,0




DUPLICATE RECORDS
0


DATASET INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Patient ID         1000 non-null   object 
 1   Name               1000 non-null   object 
 2   Date of Birth      1000 non-null   object 
 3   Gender             1000 non-null   object 
 4   Medical Condition  1000 non-null   object 
 5   Treatments         1000 non-null   object 
 6   Doctor's Notes     1000 non-null   object 
 7   Admit Date         1000 non-null   object 
 8   Discharge Date     1000 non-null   object 
 9   Bill Amount        1000 non-null   float64
dtypes: float64(1), object(9)
memory usage: 78.3+ KB


In [ ]:
# ==========================================================
# CELL 4 : DATA CLEANING
# ==========================================================

print("="*60)
print("DATA CLEANING")
print("="*60)

# Remove duplicate rows

df.drop_duplicates(inplace=True)

# Remove leading and trailing spaces from column names

df.columns = df.columns.str.strip()

# Fill missing doctor's notes

if "Doctor's Notes" in df.columns:
    df["Doctor's Notes"] = df["Doctor's Notes"].fillna("No clinical notes available.")

# Fill missing treatments

if "Treatments" in df.columns:
    df["Treatments"] = df["Treatments"].fillna("Treatment not available.")

# Fill missing medical condition

if "Medical Condition" in df.columns:
    df["Medical Condition"] = df["Medical Condition"].fillna("Unknown")

# Convert date columns

if "Admit Date" in df.columns:
    df["Admit Date"] = pd.to_datetime(df["Admit Date"])

if "Discharge Date" in df.columns:
    df["Discharge Date"] = pd.to_datetime(df["Discharge Date"])

print("Cleaning Completed Successfully!")

print("\nRemaining Missing Values")

display(df.isnull().sum())

DATA CLEANING
Cleaning Completed Successfully!

Remaining Missing Values


,0
Patient ID,0
Name,0
Date of Birth,0
Gender,0
Medical Condition,0
Treatments,0
Doctor's Notes,0
Admit Date,0
Discharge Date,0
Bill Amount,0


In [ ]:
# ==========================================================
# CELL 5 : FEATURE ENGINEERING
# ==========================================================

print("="*60)
print("FEATURE ENGINEERING")
print("="*60)

# Generate Patient ID if missing

if "Patient ID" not in df.columns:

    df.insert(
        0,
        "Patient ID",
        [
            f"PT{str(i+1).zfill(6)}"
            for i in range(len(df))
        ]
    )

# Convert DOB

if "Date of Birth" in df.columns:

    df["Date of Birth"] = pd.to_datetime(df["Date of Birth"])

    today = pd.Timestamp.today()

    df["Age"] = (
        (today - df["Date of Birth"]).dt.days // 365
    )

# Length of Stay

if (
    "Admit Date" in df.columns
    and
    "Discharge Date" in df.columns
):

    df["Length of Stay"] = (

        df["Discharge Date"]

        -

        df["Admit Date"]

    ).dt.days

# Admission Year

if "Admit Date" in df.columns:

    df["Admission Year"] = df["Admit Date"].dt.year

    df["Admission Month"] = df["Admit Date"].dt.month_name()

print("Feature Engineering Completed!")

display(df.head())

FEATURE ENGINEERING
Feature Engineering Completed!


,Patient ID,Name,Date of Birth,Gender,Medical Condition,Treatments,Doctor's Notes,Admit Date,Discharge Date,Bill Amount,Age,Length of Stay,Admission Year,Admission Month
0,921d93d1-17b8-426f-abae-5b16c7e5cd93,Debra Griffith,1971-04-01,Female,Chronic Obstructive Pulmonary Disease,Medication,Patient requires home oxygen therapy.,2021-07-12,2021-07-13,14651.34,55,1,2021,July
1,c3e70ecd-b794-4e9e-8136-ccf4ff363fe0,Robert Ross,1957-02-20,Female,Migraine,Pain Relief Medication,Follow-up to assess effectiveness of treatment.,2021-07-13,2021-08-09,718.94,69,27,2021,July
2,ef85177f-f933-4f60-aefe-6979e35c709a,Rachel Perez,1987-11-27,Female,Urinary Tract Infection,Hydration,Monitor for any signs of worsening symptoms.,2021-07-14,2021-08-06,1765.14,38,23,2021,July
3,8e36ba50-ebde-4b68-81b9-3cdca0b7eade,Michael Williams,2021-12-06,Male,Stroke,Physical Therapy,Referred to physical therapy for mobility improvement.,2021-07-15,2021-08-05,21413.36,4,21,2021,July
4,422bd94a-41e1-4bdc-96a3-f7570a868df1,Christopher Rodriguez,1990-12-19,Male,Hypertension,Lifestyle Changes,Patient started on antihypertensive medication.,2021-07-16,2021-07-20,4879.91,35,4,2021,July


In [ ]:
# ==========================================================
# CELL 6 : TEXT PREPROCESSING
# ==========================================================
#vocal interface !!!!!
lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words("english"))

def preprocess_text(text):

    text = str(text).lower()

    text = re.sub(r"\d+"," ",text)

    text = text.translate(
        str.maketrans(
            "",
            "",
            string.punctuation
        )
    )

    words = text.split()

    words = [

        lemmatizer.lemmatize(word)

        for word in words

        if word not in stop_words

    ]

    return " ".join(words)

print("="*60)
print("Cleaning Doctor Notes")
print("="*60)

df["Processed Notes"] = (

    df["Doctor's Notes"]

    .astype(str)

    .apply(preprocess_text)

)

display(

    df[

        [

            "Doctor's Notes",

            "Processed Notes"

        ]

    ].head()

)

Cleaning Doctor Notes


,Doctor's Notes,Processed Notes
0,Patient requires home oxygen therapy.,patient requires home oxygen therapy
1,Follow-up to assess effectiveness of treatment.,followup assess effectiveness treatment
2,Monitor for any signs of worsening symptoms.,monitor sign worsening symptom
3,Referred to physical therapy for mobility improvement.,referred physical therapy mobility improvement
4,Patient started on antihypertensive medication.,patient started antihypertensive medication


In [ ]:
# ==========================================================
# CELL 7 : TF-IDF VECTORIZATION
# ==========================================================

print("=" * 70)
print("BUILDING TF-IDF VECTORIZER")
print("=" * 70)

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000,
    ngram_range=(1,2)
)

tfidf_matrix = vectorizer.fit_transform(df["Processed Notes"])

print("\nTF-IDF Matrix Shape")
print(tfidf_matrix.shape)

print("\nVocabulary Size")
print(len(vectorizer.vocabulary_))

print("\nTF-IDF Vectorizer Created Successfully!")

BUILDING TF-IDF VECTORIZER

TF-IDF Matrix Shape
(1000, 458)

Vocabulary Size
458

TF-IDF Vectorizer Created Successfully!


In [ ]:
# ==========================================================
# CELL 8 : COSINE SIMILARITY MATRIX
# ==========================================================

print("=" * 70)
print("COMPUTING COSINE SIMILARITY")
print("=" * 70)

cosine_sim = cosine_similarity(tfidf_matrix)

print("Cosine Similarity Matrix Shape:")
print(cosine_sim.shape)

print("\nCosine Similarity Matrix Generated Successfully!")

COMPUTING COSINE SIMILARITY
Cosine Similarity Matrix Shape:
(1000, 1000)

Cosine Similarity Matrix Generated Successfully!


In [ ]:
# ==========================================================
# CELL 9 : INFORMATION RETRIEVAL ENGINE
# ==========================================================

def retrieve_similar_patients(patient_id, top_k=5):

    if patient_id not in df["Patient ID"].values:
        print("Patient ID not found!")
        return None

    index = df[df["Patient ID"] == patient_id].index[0]

    similarity_scores = list(enumerate(cosine_sim[index]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_k+1]

    indices = [i[0] for i in similarity_scores]

    result = df.loc[
        indices,
        [
            "Patient ID",
            "Medical Condition",
            "Treatments",
            "Doctor's Notes"
        ]
    ].copy()

    result["Similarity Score"] = [
        round(score[1]*100,2)
        for score in similarity_scores
    ]

    return result
retrieve_similar_patients("921d93d1-17b8-426f-abae-5b16c7e5cd93")

,Patient ID,Medical Condition,Treatments,Doctor's Notes,Similarity Score
66,75cdc78e-03fa-4296-9b3e-ddbe7f482f1c,Chronic Obstructive Pulmonary Disease,Medication,Patient requires home oxygen therapy.,100.0
173,26b663f5-7ea9-411f-ab22-30f8afbe51db,Chronic Obstructive Pulmonary Disease,Medication,Patient requires home oxygen therapy.,100.0
418,c754c929-71f4-47f2-b8fa-741ee84a4829,Chronic Obstructive Pulmonary Disease,Medication,Patient requires home oxygen therapy.,100.0
450,3b0b5b91-ce6b-4032-80b8-76e54a8f7c29,Chronic Obstructive Pulmonary Disease,Medication,Patient requires home oxygen therapy.,100.0
504,566af25d-dc49-472f-8728-c5eda762b23f,Chronic Obstructive Pulmonary Disease,Oxygen Therapy,Patient requires home oxygen therapy.,100.0


In [ ]:
# ==========================================================
# CELL 10 : AI CLINICAL SUMMARY
# ==========================================================

def generate_summary(patient_id):

    patient = df[df["Patient ID"] == patient_id]

    if len(patient)==0:
        return "Patient Not Found"

    patient = patient.iloc[0]

    summary = f"""

PATIENT SUMMARY

Patient ID : {patient['Patient ID']}

Medical Condition :
{patient['Medical Condition']}

Treatment :
{patient['Treatments']}

Doctor Notes :

{patient["Doctor's Notes"][:400]}

"""

    return summary
print(generate_summary("921d93d1-17b8-426f-abae-5b16c7e5cd93"))



PATIENT SUMMARY

Patient ID : 921d93d1-17b8-426f-abae-5b16c7e5cd93

Medical Condition :
Chronic Obstructive Pulmonary Disease

Treatment :
Medication

Doctor Notes :

Patient requires home oxygen therapy.




In [ ]:
# ==========================================================
# CELL 11 : DIAGNOSTIC TEST RECOMMENDATION
# ==========================================================

TEST_DATABASE = {

"Pneumonia":[
"Chest X-Ray",
"CBC",
"CRP",
"Sputum Culture"
],

"Diabetes":[
"HbA1c",
"Fasting Blood Sugar",
"OGTT"
],

"Hypertension":[
"Blood Pressure",
"ECG",
"Kidney Function Test"
],

"Heart Disease":[
"ECG",
"Echocardiogram",
"Troponin Test"
],

"Asthma":[
"Spirometry",
"Peak Flow Test"
],

"COPD":[
"Spirometry",
"Chest X-Ray"
]

}

def recommend_tests(patient_id):

    patient = df[df["Patient ID"]==patient_id]

    if len(patient)==0:
        return []

    disease = patient.iloc[0]["Medical Condition"]

    return TEST_DATABASE.get(
        disease,
        ["Consult Specialist"]
    )
recommend_tests("921d93d1-17b8-426f-abae-5b16c7e5cd93")

['Consult Specialist']

In [ ]:
# ==========================================================
# CELL 12 : COMPLETE PATIENT HISTORY
# ==========================================================

def patient_history(patient_id):

    patient = df[df["Patient ID"]==patient_id]

    if len(patient)==0:
        print("Patient Not Found")
        return

    display(patient)

    print("\n")

    print("="*70)
    print("AI SUMMARY")
    print("="*70)

    print(generate_summary(patient_id))

    print("="*70)
    print("RECOMMENDED TESTS")
    print("="*70)

    for test in recommend_tests(patient_id):

        print("•",test)

    print("\n")

    print("="*70)
    print("SIMILAR PATIENTS")
    print("="*70)

    display(
        retrieve_similar_patients(patient_id)
    )
patient_history("921d93d1-17b8-426f-abae-5b16c7e5cd93")

,Patient ID,Name,Date of Birth,Gender,Medical Condition,Treatments,Doctor's Notes,Admit Date,Discharge Date,Bill Amount,Age,Length of Stay,Admission Year,Admission Month,Processed Notes
0,921d93d1-17b8-426f-abae-5b16c7e5cd93,Debra Griffith,1971-04-01,Female,Chronic Obstructive Pulmonary Disease,Medication,Patient requires home oxygen therapy.,2021-07-12,2021-07-13,14651.34,55,1,2021,July,patient requires home oxygen therapy




AI SUMMARY


PATIENT SUMMARY

Patient ID : 921d93d1-17b8-426f-abae-5b16c7e5cd93

Medical Condition :
Chronic Obstructive Pulmonary Disease

Treatment :
Medication

Doctor Notes :

Patient requires home oxygen therapy.


RECOMMENDED TESTS
• Consult Specialist


SIMILAR PATIENTS


,Patient ID,Medical Condition,Treatments,Doctor's Notes,Similarity Score
66,75cdc78e-03fa-4296-9b3e-ddbe7f482f1c,Chronic Obstructive Pulmonary Disease,Medication,Patient requires home oxygen therapy.,100.0
173,26b663f5-7ea9-411f-ab22-30f8afbe51db,Chronic Obstructive Pulmonary Disease,Medication,Patient requires home oxygen therapy.,100.0
418,c754c929-71f4-47f2-b8fa-741ee84a4829,Chronic Obstructive Pulmonary Disease,Medication,Patient requires home oxygen therapy.,100.0
450,3b0b5b91-ce6b-4032-80b8-76e54a8f7c29,Chronic Obstructive Pulmonary Disease,Medication,Patient requires home oxygen therapy.,100.0
504,566af25d-dc49-472f-8728-c5eda762b23f,Chronic Obstructive Pulmonary Disease,Oxygen Therapy,Patient requires home oxygen therapy.,100.0


In [ ]:
# ==========================================================
# CELL 13 : SAVE COMPLETE ML PIPELINE
# ==========================================================

OUTPUT_DIR = "Doctor_AI_Project"

MODEL_DIR = os.path.join(
    OUTPUT_DIR,
    "Models"
)

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)

joblib.dump(

    vectorizer,

    os.path.join(
        MODEL_DIR,
        "tfidf_vectorizer.pkl"
    )

)

joblib.dump(

    cosine_sim,

    os.path.join(
        MODEL_DIR,
        "cosine_similarity.pkl"
    )

)

joblib.dump(

    df,

    os.path.join(
        MODEL_DIR,
        "processed_dataset.pkl"
    )

)

joblib.dump(

    TEST_DATABASE,

    os.path.join(
        MODEL_DIR,
        "test_database.pkl"
    )

)

print("="*70)
print("PROJECT SAVED SUCCESSFULLY")
print("="*70)

print("Saved Files")

print("✔ TF-IDF Vectorizer")

print("✔ Cosine Similarity")

print("✔ Processed Dataset")

print("✔ Test Recommendation Database")

PROJECT SAVED SUCCESSFULLY
Saved Files
✔ TF-IDF Vectorizer
✔ Cosine Similarity
✔ Processed Dataset
✔ Test Recommendation Database


In [ ]:
# ==========================================================
# CELL 14 : CREATE MEDICAL KNOWLEDGE BASE
# ==========================================================

print("="*70)
print("CREATING MEDICAL KNOWLEDGE BASE")
print("="*70)

knowledge_base = pd.DataFrame({

    "Medical Condition":[

        "Pneumonia",
        "Diabetes",
        "Hypertension",
        "Heart Disease",
        "Asthma",
        "COPD"

    ],

    "Recommended Tests":[

        "Chest X-Ray, CBC, CRP, Sputum Culture",

        "HbA1c, Fasting Blood Sugar, OGTT",

        "Blood Pressure, ECG, Kidney Function Test",

        "ECG, Echocardiogram, Troponin Test",

        "Spirometry, Peak Flow Test",

        "Spirometry, Chest X-Ray"

    ],

    "Priority":[

        "High",
        "Medium",
        "Medium",
        "High",
        "Medium",
        "High"

    ]

})

knowledge_base.to_csv(
    "medical_knowledge_base.csv",
    index=False
)

display(knowledge_base)

print("\nKnowledge Base Saved Successfully!")

CREATING MEDICAL KNOWLEDGE BASE


,Medical Condition,Recommended Tests,Priority
0,Pneumonia,"Chest X-Ray, CBC, CRP, Sputum Culture",High
1,Diabetes,"HbA1c, Fasting Blood Sugar, OGTT",Medium
2,Hypertension,"Blood Pressure, ECG, Kidney Function Test",Medium
3,Heart Disease,"ECG, Echocardiogram, Troponin Test",High
4,Asthma,"Spirometry, Peak Flow Test",Medium
5,COPD,"Spirometry, Chest X-Ray",High



Knowledge Base Saved Successfully!


In [ ]:
# ==========================================================
# CELL 15 : ADVANCED TEST RECOMMENDATION ENGINE
# ==========================================================

knowledge_base = pd.read_csv(
    "medical_knowledge_base.csv"
)

def recommend_tests_v2(patient_id):

    patient = df[df["Patient ID"] == patient_id]

    if patient.empty:

        return "Patient Not Found"

    disease = patient.iloc[0]["Medical Condition"]

    recommendation = knowledge_base[

        knowledge_base["Medical Condition"]

        .str.lower()

        ==

        str(disease).lower()

    ]

    if recommendation.empty:

        return pd.DataFrame({

            "Recommended Tests":["Consult Specialist"],

            "Priority":["Unknown"]

        })

    return recommendation[
        [

            "Recommended Tests",

            "Priority"

        ]
    ]

print("="*70)
print("Advanced Recommendation Engine Ready")
print("="*70)

Advanced Recommendation Engine Ready


In [ ]:
# ==========================================================
# CELL 16 : FREE TEXT CLINICAL SEARCH
# ==========================================================

def search_patient(query, top_k=5):

    processed_query = preprocess_text(query)

    query_vector = vectorizer.transform([processed_query])

    similarity = cosine_similarity(
        query_vector,
        tfidf_matrix
    )

    scores = list(
        enumerate(similarity[0])
    )

    scores = sorted(

        scores,

        key=lambda x:x[1],

        reverse=True

    )

    top = scores[:top_k]

    indices = [

        i[0]

        for i in top

    ]

    result = df.loc[

        indices,

        [

            "Patient ID",

            "Medical Condition",

            "Doctor's Notes"

        ]

    ].copy()

    result["Similarity Score"] = [

        round(i[1]*100,2)

        for i in top

    ]

    return result

print("="*70)
print("Free Text Search Engine Ready")
print("="*70)
search_patient("persistent cough with fever")

Free Text Search Engine Ready


,Patient ID,Medical Condition,Doctor's Notes,Similarity Score
84,e0aabdd4-930e-4f52-bfaf-bc8b79049286,Bronchitis,Prescribed cough medicine and advised rest.,38.6
99,b85bd9c6-5ce1-46b5-8f6f-c00f3a7c5e28,Bronchitis,Prescribed cough medicine and advised rest.,38.6
121,1480d4d6-85f2-4609-8dd3-b66e442496ed,Bronchitis,Prescribed cough medicine and advised rest.,38.6
256,2c4c0dad-96f5-430f-8579-1b83d3e8aa04,Bronchitis,Prescribed cough medicine and advised rest.,38.6
260,f1e12eb0-d89c-4eea-8b00-c15c41bd81a5,Bronchitis,Prescribed cough medicine and advised rest.,38.6


In [ ]:
# ==========================================================
# CELL 17 : IMPROVED AI CLINICAL SUMMARY
# ==========================================================

def ai_summary(patient_id):

    patient = df[df["Patient ID"]==patient_id]

    if patient.empty:

        return "Patient Not Found"

    patient = patient.iloc[0]

    notes = patient["Doctor's Notes"]

    if len(notes) > 500:

        notes = notes[:500] + "..."

    summary = {

        "Patient ID":patient["Patient ID"],

        "Medical Condition":patient["Medical Condition"],

        "Admission Date":patient["Admit Date"],

        "Discharge Date":patient["Discharge Date"],

        "Treatment":patient["Treatments"],

        "Clinical Summary":notes

    }

    return summary

print("="*70)
print("AI Summary Module Ready")
print("="*70)
summary = ai_summary("921d93d1-17b8-426f-abae-5b16c7e5cd93")
summary

AI Summary Module Ready


{'Patient ID': '921d93d1-17b8-426f-abae-5b16c7e5cd93',
 'Medical Condition': 'Chronic Obstructive Pulmonary Disease',
 'Admission Date': Timestamp('2021-07-12 00:00:00'),
 'Discharge Date': Timestamp('2021-07-13 00:00:00'),
 'Treatment': 'Medication',
 'Clinical Summary': 'Patient requires home oxygen therapy.'}

In [ ]:
# ==========================================================
# CELL 18 : KEYWORD EXTRACTION
# ==========================================================

from collections import Counter

def extract_keywords(patient_id, top_n=10):

    patient = df[df["Patient ID"]==patient_id]

    if patient.empty:

        return []

    text = patient.iloc[0]["Processed Notes"]

    words = text.split()

    frequency = Counter(words)

    keywords = frequency.most_common(top_n)

    keyword_df = pd.DataFrame(

        keywords,

        columns=[

            "Keyword",

            "Frequency"

        ]

    )

    return keyword_df

print("="*70)
print("Keyword Extraction Module Ready")
print("="*70)
extract_keywords("921d93d1-17b8-426f-abae-5b16c7e5cd93")

Keyword Extraction Module Ready


,Keyword,Frequency
0,patient,1
1,requires,1
2,home,1
3,oxygen,1
4,therapy,1


In [ ]:
# ==========================================================
# CELL 19 : SIMILARITY CONFIDENCE ENGINE
# ==========================================================

def calculate_confidence(similarity_score):

    if similarity_score >= 90:
        return "Very High"

    elif similarity_score >= 75:
        return "High"

    elif similarity_score >= 60:
        return "Medium"

    elif similarity_score >= 40:
        return "Low"

    else:
        return "Very Low"


def retrieve_similar_patients_v2(patient_id, top_k=5):

    if patient_id not in df["Patient ID"].values:

        print("Patient ID Not Found")

        return None

    index = df[df["Patient ID"] == patient_id].index[0]

    scores = list(enumerate(cosine_sim[index]))

    scores = sorted(

        scores,

        key=lambda x: x[1],

        reverse=True

    )[1:top_k+1]

    indices = [i[0] for i in scores]

    result = df.loc[

        indices,

        [

            "Patient ID",

            "Medical Condition",

            "Treatments"

        ]

    ].copy()

    result["Similarity (%)"] = [

        round(i[1]*100,2)

        for i in scores

    ]

    result["Confidence"] = [

        calculate_confidence(i[1]*100)

        for i in scores

    ]

    return result

print("Similarity Confidence Engine Ready")
retrieve_similar_patients_v2("921d93d1-17b8-426f-abae-5b16c7e5cd93")

Similarity Confidence Engine Ready


,Patient ID,Medical Condition,Treatments,Similarity (%),Confidence
66,75cdc78e-03fa-4296-9b3e-ddbe7f482f1c,Chronic Obstructive Pulmonary Disease,Medication,100.0,Very High
173,26b663f5-7ea9-411f-ab22-30f8afbe51db,Chronic Obstructive Pulmonary Disease,Medication,100.0,Very High
418,c754c929-71f4-47f2-b8fa-741ee84a4829,Chronic Obstructive Pulmonary Disease,Medication,100.0,Very High
450,3b0b5b91-ce6b-4032-80b8-76e54a8f7c29,Chronic Obstructive Pulmonary Disease,Medication,100.0,Very High
504,566af25d-dc49-472f-8728-c5eda762b23f,Chronic Obstructive Pulmonary Disease,Oxygen Therapy,100.0,Very High


In [ ]:
# ==========================================================
# CELL 20 : PATIENT VISIT TIMELINE
# ==========================================================

def patient_visit_timeline(patient_id):

    patient = df[

        df["Patient ID"] == patient_id

    ]

    if patient.empty:

        print("Patient Not Found")

        return

    timeline = patient[[

        "Patient ID",

        "Admit Date",

        "Discharge Date",

        "Medical Condition",

        "Treatments"

    ]].sort_values(

        by="Admit Date"

    )

    display(timeline)

print("Patient Timeline Module Ready")
patient_visit_timeline("921d93d1-17b8-426f-abae-5b16c7e5cd93")

Patient Timeline Module Ready


,Patient ID,Admit Date,Discharge Date,Medical Condition,Treatments
0,921d93d1-17b8-426f-abae-5b16c7e5cd93,2021-07-12,2021-07-13,Chronic Obstructive Pulmonary Disease,Medication


In [ ]:
# ==========================================================
# CELL 21 : ADVANCED PATIENT HISTORY
# ==========================================================

def advanced_patient_history(patient_id):

    patient = df[

        df["Patient ID"] == patient_id

    ]

    if patient.empty:

        print("Patient Not Found")

        return

    print("="*70)

    print("PATIENT PROFILE")

    print("="*70)

    display(patient)

    print("\n")

    print("="*70)

    print("AI SUMMARY")

    print("="*70)

    print(ai_summary(patient_id))

    print("\n")

    print("="*70)

    print("RECOMMENDED TESTS")

    print("="*70)

    display(

        recommend_tests_v2(patient_id)

    )

    print("\n")

    print("="*70)

    print("SIMILAR PATIENTS")

    print("="*70)

    display(

        retrieve_similar_patients_v2(patient_id)

    )

    print("\n")

    print("="*70)

    print("TOP KEYWORDS")

    print("="*70)

    display(

        extract_keywords(patient_id)

    )

print("Advanced History Module Ready")
advanced_patient_history("921d93d1-17b8-426f-abae-5b16c7e5cd93")

Advanced History Module Ready
PATIENT PROFILE


,Patient ID,Name,Date of Birth,Gender,Medical Condition,Treatments,Doctor's Notes,Admit Date,Discharge Date,Bill Amount,Age,Length of Stay,Admission Year,Admission Month,Processed Notes
0,921d93d1-17b8-426f-abae-5b16c7e5cd93,Debra Griffith,1971-04-01,Female,Chronic Obstructive Pulmonary Disease,Medication,Patient requires home oxygen therapy.,2021-07-12,2021-07-13,14651.34,55,1,2021,July,patient requires home oxygen therapy




AI SUMMARY
{'Patient ID': '921d93d1-17b8-426f-abae-5b16c7e5cd93', 'Medical Condition': 'Chronic Obstructive Pulmonary Disease', 'Admission Date': Timestamp('2021-07-12 00:00:00'), 'Discharge Date': Timestamp('2021-07-13 00:00:00'), 'Treatment': 'Medication', 'Clinical Summary': 'Patient requires home oxygen therapy.'}


RECOMMENDED TESTS


,Recommended Tests,Priority
0,Consult Specialist,Unknown




SIMILAR PATIENTS


,Patient ID,Medical Condition,Treatments,Similarity (%),Confidence
66,75cdc78e-03fa-4296-9b3e-ddbe7f482f1c,Chronic Obstructive Pulmonary Disease,Medication,100.0,Very High
173,26b663f5-7ea9-411f-ab22-30f8afbe51db,Chronic Obstructive Pulmonary Disease,Medication,100.0,Very High
418,c754c929-71f4-47f2-b8fa-741ee84a4829,Chronic Obstructive Pulmonary Disease,Medication,100.0,Very High
450,3b0b5b91-ce6b-4032-80b8-76e54a8f7c29,Chronic Obstructive Pulmonary Disease,Medication,100.0,Very High
504,566af25d-dc49-472f-8728-c5eda762b23f,Chronic Obstructive Pulmonary Disease,Oxygen Therapy,100.0,Very High




TOP KEYWORDS


,Keyword,Frequency
0,patient,1
1,requires,1
2,home,1
3,oxygen,1
4,therapy,1


In [ ]:
# ==========================================================
# CELL 22 : SEARCH BY MEDICAL CONDITION
# ==========================================================

def search_by_condition(condition):

    result = df[

        df["Medical Condition"]

        .str.contains(

            condition,

            case=False,

            na=False

        )

    ]

    print("Patients Found :", len(result))

    return result

print("Search By Disease Ready")
search_by_condition("Pneumonia")

Search By Disease Ready
Patients Found : 32


,Patient ID,Name,Date of Birth,Gender,Medical Condition,Treatments,Doctor's Notes,Admit Date,Discharge Date,Bill Amount,Age,Length of Stay,Admission Year,Admission Month,Processed Notes
116,dc12b127-960a-46b0-80c9-ac2836e8c0a4,Brian Sanders,2023-08-25,Male,Pneumonia,Oxygen Therapy,Monitor for any signs of respiratory distress.,2021-11-10,2021-11-30,19103.60,2,20,2021,November,monitor sign respiratory distress
224,bcbdebb8-a1b6-418b-8310-e38e1df14f27,Elizabeth Pham,2010-03-10,Female,Pneumonia,Oxygen Therapy,Started on broad-spectrum antibiotics.,2022-03-22,2022-04-10,19170.99,16,19,2022,March,started broadspectrum antibiotic
251,e4a42fc8-3508-46b1-b096-fc43a085b0c2,Jennifer Thomas,1997-10-17,Female,Pneumonia,Antibiotics,Hospitalized for closer monitoring and oxygen therapy.,2022-04-21,2022-05-02,7525.45,28,11,2022,April,hospitalized closer monitoring oxygen therapy
291,df42390e-d43f-453e-91fa-e9123d04804c,Joel Harrington,2002-08-22,Female,Pneumonia,Oxygen Therapy,Hospitalized for closer monitoring and oxygen therapy.,2022-05-30,2022-06-01,10525.40,23,2,2022,May,hospitalized closer monitoring oxygen therapy
303,592c8a9a-16ad-435a-9514-9fac5d147903,Tara Boyd,2017-02-18,Female,Pneumonia,Oxygen Therapy,Monitor for any signs of respiratory distress.,2022-06-12,2022-06-25,17521.40,9,13,2022,June,monitor sign respiratory distress
320,bccea3ec-e992-44ea-8718-b13a91fb4f37,Ryan Riley,1983-10-11,Male,Pneumonia,Hospitalization,Hospitalized for closer monitoring and oxygen therapy.,2022-07-05,2022-07-24,9695.25,42,19,2022,July,hospitalized closer monitoring oxygen therapy
364,1256e26c-562f-4a1d-a9a3-57e6e2d037bc,Robert Ramos,2017-02-09,Male,Pneumonia,Hospitalization,Hospitalized for closer monitoring and oxygen therapy.,2022-08-23,2022-09-11,16158.65,9,19,2022,August,hospitalized closer monitoring oxygen therapy
472,24d9adf5-b2d6-4f49-b84b-1b2402d943a3,Joshua Banks,1937-07-27,Female,Pneumonia,Oxygen Therapy,Started on broad-spectrum antibiotics.,2022-12-15,2022-12-21,9096.74,88,6,2022,December,started broadspectrum antibiotic
495,3220aaaa-5921-4f7c-ba71-be218d6bef59,Stephen Soto,1957-03-05,Male,Pneumonia,Antibiotics,Started on broad-spectrum antibiotics.,2023-01-10,2023-01-21,8194.67,69,11,2023,January,started broadspectrum antibiotic
527,6d0d487f-0dc0-4c11-86d4-ce002b802f75,Brandon Sherman,1927-07-29,Female,Pneumonia,Oxygen Therapy,Started on broad-spectrum antibiotics.,2023-02-10,2023-03-07,14290.38,98,25,2023,February,started broadspectrum antibiotic


In [ ]:
# ==========================================================
# CELL 23 : SEARCH BY ADMISSION DATE
# ==========================================================

def search_by_admission_date(date):

    date = pd.to_datetime(date)

    result = df[

        df["Admit Date"] == date

    ]

    print("Patients Found :", len(result))

    return result

print("Search By Admission Date Ready")
search_by_admission_date("2023-05-14")

Search By Admission Date Ready
Patients Found : 0


,Patient ID,Name,Date of Birth,Gender,Medical Condition,Treatments,Doctor's Notes,Admit Date,Discharge Date,Bill Amount,Age,Length of Stay,Admission Year,Admission Month,Processed Notes


In [ ]:
# ==========================================================
# CELL 24 : SEARCH BY GENDER
# ==========================================================

def search_by_gender(gender):

    result = df[

        df["Gender"]

        .str.lower()

        ==

        gender.lower()

    ]

    print("="*70)
    print(f"PATIENTS FOUND : {len(result)}")
    print("="*70)

    return result


print("Search By Gender Ready")
search_by_gender("Male")

Search By Gender Ready
PATIENTS FOUND : 489


,Patient ID,Name,Date of Birth,Gender,Medical Condition,Treatments,Doctor's Notes,Admit Date,Discharge Date,Bill Amount,Age,Length of Stay,Admission Year,Admission Month,Processed Notes
3,8e36ba50-ebde-4b68-81b9-3cdca0b7eade,Michael Williams,2021-12-06,Male,Stroke,Physical Therapy,Referred to physical therapy for mobility improvement.,2021-07-15,2021-08-05,21413.36,4,21,2021,July,referred physical therapy mobility improvement
4,422bd94a-41e1-4bdc-96a3-f7570a868df1,Christopher Rodriguez,1990-12-19,Male,Hypertension,Lifestyle Changes,Patient started on antihypertensive medication.,2021-07-16,2021-07-20,4879.91,35,4,2021,July,patient started antihypertensive medication
11,0df01bb8-b9b8-4d91-a5ac-e3e744286d7d,Kelly Roberts,1938-02-11,Male,Common Cold,Hydration,Symptoms should improve within a week.,2021-07-29,2021-08-28,333.02,88,30,2021,July,symptom improve within week
20,b1e3fe8e-d947-4dc0-ad4c-8bcc34cd0169,Kathleen Cohen,1962-01-09,Male,COVID-19,Antiviral Drugs,Monitor for any signs of respiratory failure.,2021-08-08,2021-08-27,15423.02,64,19,2021,August,monitor sign respiratory failure
21,b86853e7-b4b0-44f7-a5e4-b735be42da28,Stephanie Brown,1986-10-05,Male,Urinary Tract Infection,Hydration,Started on antibiotics to treat infection.,2021-08-11,2021-09-04,2517.74,39,24,2021,August,started antibiotic treat infection
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
988,2d81b5e6-5af6-4552-9a4d-8f369434f864,Jack Johnson,1929-08-23,Male,Anxiety,Cognitive Behavioral Therapy,Discussed stress management techniques.,2024-06-24,2024-06-28,3618.68,96,4,2024,June,discussed stress management technique
991,3045d288-63a1-4fb5-b981-1a894198b9c1,Rachel Serrano,1943-05-18,Male,Fracture,Physical Therapy,Fracture immobilized with casting.,2024-06-30,2024-07-21,7315.90,83,21,2024,June,fracture immobilized casting
993,1572ddfe-5431-4123-9ef5-deeac4c0e0fb,Charles Gonzalez,1995-06-26,Male,Fracture,Casting,Fracture immobilized with casting.,2024-07-02,2024-07-22,8588.94,31,20,2024,July,fracture immobilized casting
994,90bd151e-48a4-4069-9ab4-87905e646bce,Aaron Lang,1996-05-03,Male,COVID-19,Oxygen Therapy,Patient tested positive for COVID-19; started on antiviral drugs.,2024-07-03,2024-07-16,11302.16,30,13,2024,July,patient tested positive covid started antiviral drug


In [ ]:
# ==========================================================
# CELL 25 : SEARCH LOGGING
# ==========================================================

from datetime import datetime

LOG_FILE = "doctor_search_logs.csv"


def log_search(

    doctor_id,

    patient_id,

    search_type

):

    log = pd.DataFrame({

        "Timestamp":[datetime.now()],

        "Doctor ID":[doctor_id],

        "Patient ID":[patient_id],

        "Search Type":[search_type]

    })

    if os.path.exists(LOG_FILE):

        log.to_csv(

            LOG_FILE,

            mode="a",

            header=False,

            index=False

        )

    else:

        log.to_csv(

            LOG_FILE,

            index=False

        )


print("Logging Module Ready")
log_search(

    "DOC001",

    "921d93d1-17b8-426f-abae-5b16c7e5cd93",

    "Patient Search"

)

Logging Module Ready


In [ ]:
# ==========================================================
# CELL 26 : LOAD COMPLETE BACKEND
# ==========================================================

def load_backend():

    global vectorizer
    global cosine_sim
    global knowledge_base

    vectorizer = joblib.load(

        "Doctor_AI_Project/Models/tfidf_vectorizer.pkl"

    )

    cosine_sim = joblib.load(

        "Doctor_AI_Project/Models/cosine_similarity.pkl"

    )

    knowledge_base = pd.read_csv(

        "medical_knowledge_base.csv"

    )

    print("="*70)
    print("BACKEND LOADED SUCCESSFULLY")
    print("="*70)


load_backend()

BACKEND LOADED SUCCESSFULLY


In [ ]:
# ==========================================================
# CELL 27 : EXPLAINABLE AI
# ==========================================================

def explain_recommendation(patient_id):

    patient = df[

        df["Patient ID"] == patient_id

    ]

    if patient.empty:

        return "Patient Not Found"

    patient = patient.iloc[0]

    disease = patient["Medical Condition"]

    tests = recommend_tests_v2(patient_id)

    explanation = {

        "Medical Condition": disease,

        "Reason":
        f"The patient is diagnosed with {disease}. Recommended diagnostic tests follow standard clinical practice for this condition.",

        "Recommended Tests": tests

    }

    return explanation


print("Explainable AI Ready")
explain_recommendation(
"921d93d1-17b8-426f-abae-5b16c7e5cd93"
)

Explainable AI Ready


{'Medical Condition': 'Chronic Obstructive Pulmonary Disease',
 'Reason': 'The patient is diagnosed with Chronic Obstructive Pulmonary Disease. Recommended diagnostic tests follow standard clinical practice for this condition.',
 'Recommended Tests':     Recommended Tests Priority
 0  Consult Specialist  Unknown}

In [ ]:
# ==========================================================
# CELL 28 : PATIENT RISK INDICATOR
# ==========================================================

HIGH_RISK = [

    "Pneumonia",

    "Heart Disease",

    "COPD"

]

MEDIUM_RISK = [

    "Asthma",

    "Hypertension"

]

LOW_RISK = [

    "Diabetes"

]


def risk_indicator(patient_id):

    patient = df[

        df["Patient ID"] == patient_id

    ]

    if patient.empty:

        return "Patient Not Found"

    disease = patient.iloc[0]["Medical Condition"]

    if disease in HIGH_RISK:

        return "🔴 HIGH RISK"

    elif disease in MEDIUM_RISK:

        return "🟡 MEDIUM RISK"

    elif disease in LOW_RISK:

        return "🟢 LOW RISK"

    else:

        return "⚪ UNKNOWN"


print("Risk Indicator Ready")
risk_indicator(
"921d93d1-17b8-426f-abae-5b16c7e5cd93"
)

Risk Indicator Ready


'⚪ UNKNOWN'

In [ ]:
# ==========================================================
# CELL 29 : DASHBOARD ANALYTICS
# ==========================================================

print("="*70)
print("DOCTOR DASHBOARD ANALYTICS")
print("="*70)

print("\nTotal Patients")
print(len(df))

print("\nGender Distribution")

display(

    df["Gender"]

    .value_counts()

)

print("\nTop Medical Conditions")

display(

    df["Medical Condition"]

    .value_counts()

    .head(10)

)

print("\nAverage Bill Amount")

if "Bill Amount" in df.columns:

    print(

        round(

            df["Bill Amount"].mean(),

            2

        )

    )

print("\nAverage Length of Stay")

if "Length of Stay" in df.columns:

    print(

        round(

            df["Length of Stay"]

            .mean(),

            2

        ),

        "Days"

    )

print("\nAdmissions Per Year")

if "Admission Year" in df.columns:

    display(

        df["Admission Year"]

        .value_counts()

        .sort_index()

    )

print("="*70)
print("Dashboard Ready")
print("="*70)

DOCTOR DASHBOARD ANALYTICS

Total Patients
1000

Gender Distribution


,count
Gender,
Female,511
Male,489



Top Medical Conditions


,count
Medical Condition,
Skin Infection,46
Alzheimer's Disease,46
Migraine,43
Stroke,42
Bronchitis,41
Common Cold,40
Influenza,37
Fracture,37
Depression,37



Average Bill Amount
9590.63

Average Length of Stay
15.68 Days

Admissions Per Year


,count
Admission Year,
2021,155
2022,326
2023,352
2024,167


Dashboard Ready


In [ ]:
# ==========================================================
# CELL 30 : HYBRID SEMANTIC SEARCH ENGINE
# ==========================================================

print("="*70)
print("INSTALLING SENTENCE TRANSFORMERS")
print("="*70)

!pip -q install sentence-transformers

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("="*70)
print("LOADING PRETRAINED MODEL")
print("="*70)

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Model Loaded Successfully!")

print("="*70)
print("GENERATING EMBEDDINGS")
print("="*70)

sentence_embeddings = embedding_model.encode(

    df["Processed Notes"].tolist(),

    show_progress_bar=True,

    convert_to_numpy=True

)

print("Embedding Shape :", sentence_embeddings.shape)

print("="*70)
print("Semantic Embeddings Ready")
print("="*70)

INSTALLING SENTENCE TRANSFORMERS
LOADING PRETRAINED MODEL


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model Loaded Successfully!
GENERATING EMBEDDINGS


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Embedding Shape : (1000, 384)
Semantic Embeddings Ready


In [ ]:
# ==========================================================
# CELL 31 : HYBRID SEARCH
# ==========================================================

def hybrid_search(query, top_k=5):

    query = preprocess_text(query)

    ####################################
    # TF-IDF SCORE
    ####################################

    tfidf_query = vectorizer.transform([query])

    tfidf_scores = cosine_similarity(

        tfidf_query,

        tfidf_matrix

    )[0]

    ####################################
    # SEMANTIC SCORE
    ####################################

    query_embedding = embedding_model.encode(

        [query],

        convert_to_numpy=True

    )

    semantic_scores = cosine_similarity(

        query_embedding,

        sentence_embeddings

    )[0]

    ####################################
    # HYBRID SCORE
    ####################################

    hybrid_scores = (

        0.5 * tfidf_scores

        +

        0.5 * semantic_scores

    )

    top_indices = np.argsort(

        hybrid_scores

    )[::-1][:top_k]

    result = df.iloc[

        top_indices

    ][

        [

            "Patient ID",

            "Medical Condition",

            "Treatments",

            "Doctor's Notes"

        ]

    ].copy()

    result["Hybrid Score"] = [

        round(

            hybrid_scores[i]*100,

            2

        )

        for i in top_indices

    ]

    return result

print("Hybrid Search Engine Ready!")

Hybrid Search Engine Ready!


In [ ]:
# ==========================================================
# CELL 32 : TEST HYBRID SEARCH
# ==========================================================

hybrid_search(

    "patient with severe cough fever breathing difficulty"

)

,Patient ID,Medical Condition,Treatments,Doctor's Notes,Hybrid Score
99,b85bd9c6-5ce1-46b5-8f6f-c00f3a7c5e28,Bronchitis,Rest,Prescribed cough medicine and advised rest.,37.51
84,e0aabdd4-930e-4f52-bfaf-bc8b79049286,Bronchitis,Antibiotics,Prescribed cough medicine and advised rest.,37.51
121,1480d4d6-85f2-4609-8dd3-b66e442496ed,Bronchitis,Cough Medicine,Prescribed cough medicine and advised rest.,37.51
669,8890be68-1b55-4f89-b030-771f39d6e223,Bronchitis,Antibiotics,Prescribed cough medicine and advised rest.,37.51
857,f942053c-a613-451f-baa7-7a9f4cca2952,Bronchitis,Rest,Prescribed cough medicine and advised rest.,37.51


In [ ]:
# ==========================================================
# CELL 33 : SAVE HYBRID SEARCH ENGINE
# ==========================================================

joblib.dump(

    sentence_embeddings,

    "Doctor_AI_Project/Models/sentence_embeddings.pkl"

)

joblib.dump(

    embedding_model,

    "Doctor_AI_Project/Models/minilm_model.pkl"

)

print("="*70)
print("HYBRID AI ENGINE SAVED")
print("="*70)

print("Saved")

print("✔ Sentence Embeddings")

print("✔ MiniLM Model")

HYBRID AI ENGINE SAVED
Saved
✔ Sentence Embeddings
✔ MiniLM Model


In [ ]:
# ==========================================================
# CELL 34 : DOCTOR AI CLINICAL ASSISTANT
# ==========================================================

def doctor_ai_assistant(patient_id, display_report=True):

    # ======================================================
    # CHECK PATIENT EXISTS
    # ======================================================

    if patient_id not in df["Patient ID"].values:

        print("=" * 80)
        print("PATIENT NOT FOUND")
        print("=" * 80)

        return None

    # ======================================================
    # FETCH PATIENT RECORD
    # ======================================================

    patient = df[
        df["Patient ID"] == patient_id
    ].iloc[0]

    # ======================================================
    # AI MODULES
    # ======================================================

    summary = ai_summary(patient_id)

    similar_patients = retrieve_similar_patients_v2(
        patient_id,
        top_k=5
    )

    recommended_tests = recommend_tests_v2(
        patient_id
    )

    keywords = extract_keywords(
        patient_id
    )

    risk = risk_indicator(
        patient_id
    )

    timeline = df[
        df["Patient ID"] == patient_id
    ][
        [
            "Admit Date",
            "Discharge Date",
            "Medical Condition",
            "Treatments"
        ]
    ]

    explanation = explain_recommendation(
        patient_id
    )

    # ======================================================
    # CREATE RESPONSE OBJECT
    # ======================================================

    report = {

        "Patient Information":{

            "Patient ID":patient["Patient ID"],

            "Name":patient["Name"],

            "Gender":patient["Gender"],

            "Medical Condition":patient["Medical Condition"],

            "Treatment":patient["Treatments"],

            "Admit Date":str(patient["Admit Date"]),

            "Discharge Date":str(patient["Discharge Date"])

        },

        "AI Summary":summary,

        "Risk":risk,

        "Recommended Tests":recommended_tests,

        "Keywords":keywords,

        "Similar Patients":similar_patients,

        "Timeline":timeline,

        "Explainable AI":explanation

    }

    # ======================================================
    # DISPLAY REPORT
    # ======================================================

    if display_report:

        print("\n")
        print("=" * 90)
        print("            DOCTOR AI CLINICAL REPORT")
        print("=" * 90)

        # --------------------------------------------------
        # PATIENT INFORMATION
        # --------------------------------------------------

        print("\nPATIENT INFORMATION\n")

        display(
            pd.DataFrame(
                [report["Patient Information"]]
            )
        )

        # --------------------------------------------------
        # AI SUMMARY
        # --------------------------------------------------

        print("\nAI CLINICAL SUMMARY\n")

        if isinstance(report["AI Summary"], dict):

            display(
                pd.DataFrame(
                    [report["AI Summary"]]
                )
            )

        else:

            print(report["AI Summary"])

        # --------------------------------------------------
        # RISK
        # --------------------------------------------------

        print("\nPATIENT RISK\n")

        display(
            pd.DataFrame({

                "Risk Level":[
                    report["Risk"]
                ]

            })
        )

        # --------------------------------------------------
        # RECOMMENDED TESTS
        # --------------------------------------------------

        print("\nRECOMMENDED DIAGNOSTIC TESTS\n")

        if isinstance(report["Recommended Tests"], pd.DataFrame):

            display(
                report["Recommended Tests"]
            )

        else:

            display(
                pd.DataFrame(
                    report["Recommended Tests"]
                )
            )

        # --------------------------------------------------
        # TOP KEYWORDS
        # --------------------------------------------------

        print("\nTOP CLINICAL KEYWORDS\n")

        display(
            report["Keywords"]
        )

        # --------------------------------------------------
        # SIMILAR PATIENTS
        # --------------------------------------------------

        print("\nTOP SIMILAR PATIENTS\n")

        display(
            report["Similar Patients"]
        )

        # --------------------------------------------------
        # PATIENT TIMELINE
        # --------------------------------------------------

        print("\nVISIT HISTORY\n")

        display(
            report["Timeline"]
        )

        # --------------------------------------------------
        # EXPLAINABLE AI
        # --------------------------------------------------

        print("\nEXPLAINABLE AI\n")

        if isinstance(report["Explainable AI"], dict):

            explanation_df = pd.DataFrame({

                "Medical Condition":[
                    report["Explainable AI"]["Medical Condition"]
                ],

                "Reason":[
                    report["Explainable AI"]["Reason"]
                ]

            })

            display(explanation_df)

        else:

            print(report["Explainable AI"])

        print("=" * 90)
        print("END OF REPORT")
        print("=" * 90)

    return report
doctor_ai_assistant(
    "921d93d1-17b8-426f-abae-5b16c7e5cd93"
)
doctor_ai_assistant(
    "921d93d1-17b8-426f-abae-5b16c7e5cd93",
    display_report=False
)



            DOCTOR AI CLINICAL REPORT

PATIENT INFORMATION



,Patient ID,Name,Gender,Medical Condition,Treatment,Admit Date,Discharge Date
0,921d93d1-17b8-426f-abae-5b16c7e5cd93,Debra Griffith,Female,Chronic Obstructive Pulmonary Disease,Medication,2021-07-12 00:00:00,2021-07-13 00:00:00



AI CLINICAL SUMMARY



,Patient ID,Medical Condition,Admission Date,Discharge Date,Treatment,Clinical Summary
0,921d93d1-17b8-426f-abae-5b16c7e5cd93,Chronic Obstructive Pulmonary Disease,2021-07-12,2021-07-13,Medication,Patient requires home oxygen therapy.



PATIENT RISK



...

,Admit Date,Discharge Date,Medical Condition,Treatments
0,2021-07-12,2021-07-13,Chronic Obstructive Pulmonary Disease,Medication



EXPLAINABLE AI



,Medical Condition,Reason
0,Chronic Obstructive Pulmonary Disease,The patient is diagnosed with Chronic Obstructive Pulmonary Disease. Recommended diagnostic tests follow standard clinical practice for this condition.


END OF REPORT


{'Patient Information': {'Patient ID': '921d93d1-17b8-426f-abae-5b16c7e5cd93',
  'Name': 'Debra Griffith',
  'Gender': 'Female',
  'Medical Condition': 'Chronic Obstructive Pulmonary Disease',
  'Treatment': 'Medication',
  'Admit Date': '2021-07-12 00:00:00',
  'Discharge Date': '2021-07-13 00:00:00'},
 'AI Summary': {'Patient ID': '921d93d1-17b8-426f-abae-5b16c7e5cd93',
  'Medical Condition': 'Chronic Obstructive Pulmonary Disease',
  'Admission Date': Timestamp('2021-07-12 00:00:00'),
  'Discharge Date': Timestamp('2021-07-13 00:00:00'),
  'Treatment': 'Medication',
  'Clinical Summary': 'Patient requires home oxygen therapy.'},
 'Risk': '⚪ UNKNOWN',
 'Recommended Tests':     Recommended Tests Priority
 0  Consult Specialist  Unknown,
 'Keywords':     Keyword  Frequency
 0   patient          1
 1  requires          1
 2      home          1
 3    oxygen          1
 4   therapy          1,
 'Similar Patients':                                Patient ID  \
 66   75cdc78e-03fa-4296-9b3e

In [ ]:
# ==========================================================
# CELL 35 : PATIENT OBJECT LOADER
# ==========================================================

def get_patient(patient_id):

    """
    Fetch a patient record from the dataset.

    Returns:
        pandas.Series
    """

    patient = df[
        df["Patient ID"] == patient_id
    ]

    if patient.empty:

        return None

    return patient.iloc[0]


print("=" * 70)
print("PATIENT OBJECT LOADER READY")
print("=" * 70)
patient = get_patient(
    "921d93d1-17b8-426f-abae-5b16c7e5cd93"
)

patient

PATIENT OBJECT LOADER READY


,0
Patient ID,921d93d1-17b8-426f-abae-5b16c7e5cd93
Name,Debra Griffith
Date of Birth,1971-04-01 00:00:00
Gender,Female
Medical Condition,Chronic Obstructive Pulmonary Disease
Treatments,Medication
Doctor's Notes,Patient requires home oxygen therapy.
Admit Date,2021-07-12 00:00:00
Discharge Date,2021-07-13 00:00:00
Bill Amount,14651.34


In [ ]:
# ==========================================================
# CELL 36 : AI UTILITY FUNCTIONS
# ==========================================================

from collections import Counter


def generate_ai_summary(patient):

    notes = str(patient["Doctor's Notes"])

    if len(notes) > 500:

        notes = notes[:500] + "..."

    return {

        "Patient ID": patient["Patient ID"],

        "Patient Name": patient["Name"],

        "Medical Condition": patient["Medical Condition"],

        "Treatment": patient["Treatments"],

        "Clinical Summary": notes

    }


def generate_keywords(patient, top_n=10):

    text = preprocess_text(
        patient["Doctor's Notes"]
    )

    frequency = Counter(text.split())

    keywords = frequency.most_common(top_n)

    return pd.DataFrame(

        keywords,

        columns=[

            "Keyword",

            "Frequency"

        ]

    )


def generate_risk(patient):

    disease = patient["Medical Condition"]

    if disease in HIGH_RISK:

        return "🔴 HIGH RISK"

    elif disease in MEDIUM_RISK:

        return "🟡 MEDIUM RISK"

    elif disease in LOW_RISK:

        return "🟢 LOW RISK"

    else:

        return "⚪ UNKNOWN"


print("=" * 70)
print("AI UTILITY FUNCTIONS READY")
print("=" * 70)

AI UTILITY FUNCTIONS READY


In [ ]:
# ==========================================================
# CELL 37 : CENTRAL RECOMMENDATION ENGINE
# ==========================================================

def generate_recommendation(patient):

    disease = str(
        patient["Medical Condition"]
    )

    recommendation = knowledge_base[

        knowledge_base["Medical Condition"]

        .str.lower()

        ==

        disease.lower()

    ]

    if recommendation.empty:

        recommendation = pd.DataFrame({

            "Medical Condition":[disease],

            "Recommended Tests":[
                "Consult Specialist"
            ],

            "Priority":[
                "Unknown"
            ]

        })

    return recommendation.reset_index(drop=True)


print("=" * 70)
print("RECOMMENDATION ENGINE READY")
print("=" * 70)

RECOMMENDATION ENGINE READY


In [ ]:
# ==========================================================
# CELL 38 : COMPLETE REPORT GENERATOR
# ==========================================================

def generate_complete_report(patient_id):

    patient = get_patient(patient_id)

    if patient is None:

        return None

    report = {

        "Patient": patient,

        "Summary": generate_ai_summary(patient),

        "Keywords": generate_keywords(patient),

        "Risk": generate_risk(patient),

        "Recommendations": generate_recommendation(patient),

        "Similar Patients": retrieve_similar_patients_v2(
            patient["Patient ID"]
        ),

        "Timeline": df[
            df["Patient ID"] == patient["Patient ID"]
        ][

            [

                "Admit Date",

                "Discharge Date",

                "Medical Condition",

                "Treatments"

            ]

        ]

    }

    return report


print("=" * 70)
print("REPORT GENERATOR READY")
print("=" * 70)

REPORT GENERATOR READY


In [ ]:
# ==========================================================
# CELL 39 : FLASK READY BACKEND API
# ==========================================================

def backend_api(patient_id):

    report = generate_complete_report(
        patient_id
    )

    if report is None:

        return {

            "status": False,

            "message": "Patient Not Found"

        }

    response = {

        "status": True,

        "patient_information":

            report["Patient"].to_dict(),

        "summary":

            report["Summary"],

        "risk":

            report["Risk"],

        "recommendations":

            report["Recommendations"].to_dict(
                orient="records"
            ),

        "keywords":

            report["Keywords"].to_dict(
                orient="records"
            ),

        "similar_patients":

            report["Similar Patients"].to_dict(
                orient="records"
            ),

        "timeline":

            report["Timeline"].to_dict(
                orient="records"
            )

    }

    return response


print("=" * 70)
print("FLASK READY API CREATED")
print("=" * 70)
report = backend_api(
    "921d93d1-17b8-426f-abae-5b16c7e5cd93"
)

from pprint import pprint

print(report)

FLASK READY API CREATED
{'status': True, 'patient_information': {'Patient ID': '921d93d1-17b8-426f-abae-5b16c7e5cd93', 'Name': 'Debra Griffith', 'Date of Birth': Timestamp('1971-04-01 00:00:00'), 'Gender': 'Female', 'Medical Condition': 'Chronic Obstructive Pulmonary Disease', 'Treatments': 'Medication', "Doctor's Notes": 'Patient requires home oxygen therapy.', 'Admit Date': Timestamp('2021-07-12 00:00:00'), 'Discharge Date': Timestamp('2021-07-13 00:00:00'), 'Bill Amount': 14651.34, 'Age': 55, 'Length of Stay': 1, 'Admission Year': 2021, 'Admission Month': 'July', 'Processed Notes': 'patient requires home oxygen therapy'}, 'summary': {'Patient ID': '921d93d1-17b8-426f-abae-5b16c7e5cd93', 'Patient Name': 'Debra Griffith', 'Medical Condition': 'Chronic Obstructive Pulmonary Disease', 'Treatment': 'Medication', 'Clinical Summary': 'Patient requires home oxygen therapy.'}, 'risk': '⚪ UNKNOWN', 'recommendations': [{'Medical Condition': 'Chronic Obstructive Pulmonary Disease', 'Recommended

In [ ]:
# ==========================================================
# CELL 40 : EXPORT AI REPORT AS JSON
# ==========================================================

import json
import os

OUTPUT_FOLDER = "Doctor_AI_Project/Reports"

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)

def export_json(patient_id):

    report = backend_api(patient_id)

    if report["status"] == False:

        print("Patient Not Found")
        return

    filename = os.path.join(
        OUTPUT_FOLDER,
        f"{patient_id}_Report.json"
    )

    with open(filename, "w") as file:

        json.dump(
            report,
            file,
            indent=4,
            default=str
        )

    print("="*70)
    print("JSON REPORT SAVED")
    print(filename)
    print("="*70)
    export_json(
"921d93d1-17b8-426f-abae-5b16c7e5cd93"
)

In [ ]:
# ==========================================================
# CELL 41 : EXPORT AI REPORT AS CSV
# ==========================================================

def export_csv(patient_id):

    report = backend_api(patient_id)

    if report["status"] == False:

        print("Patient Not Found")
        return

    patient = pd.DataFrame(
        [report["patient_information"]]
    )

    summary = pd.DataFrame(
        [report["summary"]]
    )

    recommendations = pd.DataFrame(
        report["recommendations"]
    )

    similar = pd.DataFrame(
        report["similar_patients"]
    )

    patient.to_csv(
        os.path.join(
            OUTPUT_FOLDER,
            f"{patient_id}_Patient.csv"
        ),
        index=False
    )

    summary.to_csv(
        os.path.join(
            OUTPUT_FOLDER,
            f"{patient_id}_Summary.csv"
        ),
        index=False
    )

    recommendations.to_csv(
        os.path.join(
            OUTPUT_FOLDER,
            f"{patient_id}_Recommendations.csv"
        ),
        index=False
    )

    similar.to_csv(
        os.path.join(
            OUTPUT_FOLDER,
            f"{patient_id}_SimilarPatients.csv"
        ),
        index=False
    )

    print("="*70)
    print("CSV REPORTS EXPORTED")
    print("="*70)

In [ ]:
# ==========================================================
# CELL 42 : EXPORT PDF REPORT
# ==========================================================

!pip -q install reportlab

from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer
)

from reportlab.lib.styles import getSampleStyleSheet

styles = getSampleStyleSheet()

def export_pdf(patient_id):

    report = backend_api(patient_id)

    if report["status"] == False:

        print("Patient Not Found")
        return

    filename = os.path.join(

        OUTPUT_FOLDER,

        f"{patient_id}_Clinical_Report.pdf"

    )

    document = SimpleDocTemplate(
        filename
    )

    story = []

    story.append(

        Paragraph(

            "<b>DOCTOR AI CLINICAL REPORT</b>",

            styles["Title"]

        )

    )

    story.append(Spacer(1,20))

    patient = report["patient_information"]

    for key,value in patient.items():

        story.append(

            Paragraph(

                f"<b>{key}</b> : {value}",

                styles["Normal"]

            )

        )

    story.append(Spacer(1,20))

    story.append(

        Paragraph(

            "<b>AI SUMMARY</b>",

            styles["Heading2"]

        )

    )

    story.append(

        Paragraph(

            str(report["summary"]),

            styles["BodyText"]

        )

    )

    story.append(Spacer(1,20))

    story.append(

        Paragraph(

            "<b>RISK LEVEL</b>",

            styles["Heading2"]

        )

    )

    story.append(

        Paragraph(

            report["risk"],

            styles["BodyText"]

        )

    )

    story.append(Spacer(1,20))

    story.append(

        Paragraph(

            "<b>RECOMMENDED TESTS</b>",

            styles["Heading2"]

        )

    )

    for row in report["recommendations"]:

        story.append(

            Paragraph(

                str(row),

                styles["BodyText"]

            )

        )

    document.build(story)

    print("="*70)
    print("PDF REPORT GENERATED")
    print(filename)
    print("="*70)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 27.6 MB/s eta 0:00:00


In [ ]:
# ==========================================================
# CELL 43 : PATIENT STATISTICS
# ==========================================================

def patient_statistics():

    print("="*70)
    print("PATIENT STATISTICS")
    print("="*70)

    print("Total Patients :",len(df))

    print("\nGender Distribution")

    display(

        df["Gender"]

        .value_counts()

    )

    if "Age" in df.columns:

        print("\nAverage Age")

        print(

            round(

                df["Age"].mean(),

                2

            )

        )

    if "Length of Stay" in df.columns:

        print("\nAverage Stay")

        print(

            round(

                df["Length of Stay"]

                .mean(),

                2

            ),

            "Days"

        )

print("Patient Statistics Ready")

Patient Statistics Ready


In [ ]:
# ==========================================================
# CELL 44 : DISEASE ANALYTICS
# ==========================================================

def disease_statistics():

    print("="*70)
    print("DISEASE ANALYTICS")
    print("="*70)

    disease = (

        df["Medical Condition"]

        .value_counts()

    )

    display(disease)

    print("\nMost Common Disease")

    print(

        disease.idxmax()

    )

print("Disease Analytics Ready")

Disease Analytics Ready


In [ ]:
# ==========================================================
# CELL 45 : DOCTOR ACTIVITY LOGGER
# ==========================================================

ACTIVITY_FILE = "Doctor_AI_Project/doctor_activity.csv"

def log_doctor_activity(

    doctor_id,

    patient_id,

    action

):

    activity = pd.DataFrame({

        "Timestamp":[datetime.now()],

        "Doctor ID":[doctor_id],

        "Patient ID":[patient_id],

        "Action":[action]

    })

    if os.path.exists(ACTIVITY_FILE):

        activity.to_csv(

            ACTIVITY_FILE,

            mode="a",

            header=False,

            index=False

        )

    else:

        activity.to_csv(

            ACTIVITY_FILE,

            index=False

        )

    print("Activity Logged Successfully")

print("Doctor Activity Logger Ready")
log_doctor_activity(

    "DOC001",

    "921d93d1-17b8-426f-abae-5b16c7e5cd93",

    "Viewed Patient Report"

)

Doctor Activity Logger Ready
Activity Logged Successfully


In [ ]:
# ==========================================================
# CELL 46 : BACKEND CONFIGURATION
# ==========================================================

CONFIG = {

    "PROJECT_NAME":"Doctor AI Portal",

    "VERSION":"1.0",

    "AUTHOR":"Your Name",

    "MODEL_DIRECTORY":"Doctor_AI_Project/Models",

    "REPORT_DIRECTORY":"Doctor_AI_Project/Reports",

    "LOG_DIRECTORY":"Doctor_AI_Project/Logs",

    "TOP_K_RESULTS":5,

    "DEFAULT_LANGUAGE":"English"

}

print("="*70)
print("BACKEND CONFIGURATION LOADED")
print("="*70)

for key,value in CONFIG.items():

    print(f"{key} : {value}")

BACKEND CONFIGURATION LOADED
PROJECT_NAME : Doctor AI Portal
VERSION : 1.0
AUTHOR : Your Name
MODEL_DIRECTORY : Doctor_AI_Project/Models
REPORT_DIRECTORY : Doctor_AI_Project/Reports
LOG_DIRECTORY : Doctor_AI_Project/Logs
TOP_K_RESULTS : 5
DEFAULT_LANGUAGE : English


In [ ]:
# ==========================================================
# CELL 47 : VERIFY ALL REQUIRED FILES
# ==========================================================

required_files = [

    "Doctor_AI_Project/Models/tfidf_vectorizer.pkl",

    "Doctor_AI_Project/Models/cosine_similarity.pkl",

    "Doctor_AI_Project/Models/processed_dataset.pkl",

    "Doctor_AI_Project/Models/test_database.pkl",

    "Doctor_AI_Project/Models/sentence_embeddings.pkl"

]

print("="*70)
print("VERIFYING BACKEND FILES")
print("="*70)

missing=[]

for file in required_files:

    if os.path.exists(file):

        print("✔",file)

    else:

        print("✖",file)

        missing.append(file)

if len(missing)==0:

    print("\nAll backend files verified successfully.")

else:

    print("\nMissing Files")

    print(missing)

VERIFYING BACKEND FILES
✔ Doctor_AI_Project/Models/tfidf_vectorizer.pkl
✔ Doctor_AI_Project/Models/cosine_similarity.pkl
✔ Doctor_AI_Project/Models/processed_dataset.pkl
✔ Doctor_AI_Project/Models/test_database.pkl
✔ Doctor_AI_Project/Models/sentence_embeddings.pkl

All backend files verified successfully.


In [ ]:
# ==========================================================
# CELL 48 : BACKEND HEALTH CHECK
# ==========================================================

def backend_health_check():

    print("="*70)

    print("BACKEND HEALTH REPORT")

    print("="*70)

    print("Dataset Loaded :",len(df)>0)

    print("Knowledge Base :",len(knowledge_base)>0)

    print("TF-IDF Vocabulary :",len(vectorizer.vocabulary_))

    print("Similarity Matrix :",cosine_sim.shape)

    print("Sentence Embeddings :",sentence_embeddings.shape)

    print("Total Patients :",len(df))

    print("="*70)

backend_health_check()

BACKEND HEALTH REPORT
Dataset Loaded : True
Knowledge Base : True
TF-IDF Vocabulary : 458
Similarity Matrix : (1000, 1000)
Sentence Embeddings : (1000, 384)
Total Patients : 1000


In [ ]:
# ==========================================================
# CELL 49 : INITIALIZE BACKEND
# ==========================================================

def initialize_backend():

    print("="*70)
    print("INITIALIZING DOCTOR AI BACKEND")
    print("="*70)

    backend_health_check()

    print()

    print("Backend Initialized Successfully")

    print("Ready for Flask Integration")

    print("="*70)

initialize_backend()

INITIALIZING DOCTOR AI BACKEND
BACKEND HEALTH REPORT
Dataset Loaded : True
Knowledge Base : True
TF-IDF Vocabulary : 458
Similarity Matrix : (1000, 1000)
Sentence Embeddings : (1000, 384)
Total Patients : 1000

Backend Initialized Successfully
Ready for Flask Integration


In [ ]:
# ==========================================================
# CELL 50 : PROJECT STARTUP
# ==========================================================

def start_doctor_ai():

    print("="*80)

    print("         DOCTOR AI PORTAL")

    print("="*80)

    print()

    print("Version :",CONFIG["VERSION"])

    print("Project :",CONFIG["PROJECT_NAME"])

    print()

    print("Backend Status : READY")

    print("AI Engine : READY")

    print("TF-IDF : READY")

    print("Semantic Search : READY")

    print("Recommendation Engine : READY")

    print("Clinical Report Engine : READY")

    print("PDF Export : READY")

    print("Logging System : READY")

    print()

    print("="*80)

    print("SYSTEM READY")

    print("="*80)

start_doctor_ai()

         DOCTOR AI PORTAL

Version : 1.0
Project : Doctor AI Portal

Backend Status : READY
AI Engine : READY
TF-IDF : READY
Semantic Search : READY
Recommendation Engine : READY
Clinical Report Engine : READY
PDF Export : READY
Logging System : READY

SYSTEM READY


In [ ]:
import joblib
import os

os.makedirs("Doctor_AI_Project/Models", exist_ok=True)

joblib.dump(df, "Doctor_AI_Project/Models/processed_dataset.pkl")
joblib.dump(vectorizer, "Doctor_AI_Project/Models/tfidf_vectorizer.pkl")
joblib.dump(cosine_sim, "Doctor_AI_Project/Models/cosine_similarity.pkl")
joblib.dump(sentence_embeddings, "Doctor_AI_Project/Models/sentence_embeddings.pkl")
joblib.dump(embedding_model, "Doctor_AI_Project/Models/minilm_model.pkl")

knowledge_base.to_csv(
    "Doctor_AI_Project/medical_knowledge_base.csv",
    index=False
)

print("All backend files saved successfully!")

All backend files saved successfully!


In [ ]:
# ==========================================================
# SAVE BACKEND TO GOOGLE DRIVE
# ==========================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

DRIVE_PATH = "/content/drive/MyDrive/Doctor_AI_Project"

os.makedirs(DRIVE_PATH, exist_ok=True)
os.makedirs(f"{DRIVE_PATH}/Models", exist_ok=True)
os.makedirs(f"{DRIVE_PATH}/Reports", exist_ok=True)

print("Google Drive folders created successfully!")

Google Drive folders created successfully!


In [ ]:
import joblib

print("="*70)
print("SAVING BACKEND TO GOOGLE DRIVE")
print("="*70)

# Dataset
joblib.dump(
    df,
    f"{DRIVE_PATH}/Models/processed_dataset.pkl"
)

# TF-IDF Vectorizer
joblib.dump(
    vectorizer,
    f"{DRIVE_PATH}/Models/tfidf_vectorizer.pkl"
)

# Cosine Similarity Matrix
joblib.dump(
    cosine_sim,
    f"{DRIVE_PATH}/Models/cosine_similarity.pkl"
)

# Sentence Embeddings
joblib.dump(
    sentence_embeddings,
    f"{DRIVE_PATH}/Models/sentence_embeddings.pkl"
)

# MiniLM Model
joblib.dump(
    embedding_model,
    f"{DRIVE_PATH}/Models/minilm_model.pkl"
)

# Knowledge Base
knowledge_base.to_csv(
    f"{DRIVE_PATH}/medical_knowledge_base.csv",
    index=False
)

print("="*70)
print("BACKEND SAVED SUCCESSFULLY")
print("="*70)

SAVING BACKEND TO GOOGLE DRIVE
BACKEND SAVED SUCCESSFULLY
